### Installation of DeepEval

In [ ]:
# %pip install -U deepeval
# %pip install dotenv
# %pip install litellm

### Creating Confident AI login

In [18]:
import deepeval

deepeval.login("confident_us_8k9P7QpyyKgjpa7yzXG0ULlki3JAwq0DPAstgNKA1x0=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

#### Login Details. 
- this above api key is captured from app.confident-ai.com after login. It need corporatel login, so emails from temp emails can be used. After successful login it will give option to Get API key.
- Confident AI is a DeepEval product. DeepEval is a platform for evaluating and benchmarking AI models.
- I have logged in with below creds. 
- Email:levovab958@datehype.com
- Password:Test@1234  -->

In [19]:
from dotenv import load_dotenv

load = load_dotenv('./../.env')

- need openAI API key, that we can capture from https://platform.openai.com/, logging in - Setting - API keys. 
- Create a team, and it will give an option to get an API key.
- cmd -> export OPENAI_API_KEY= Captured API Key here. --> If not working, but its working directly. 

### Writing simple DeepEval Test

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""  # Clear OpenAI key
os.environ["DEEPEVAL_MODEL"] = "ollama/llama3.2:latest"

### Answer Relevancy Metrics - Standalone

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.models import OllamaModel
# from deepeval.models import GPTModel
from deepeval.metrics import AnswerRelevancyMetric
# model = GPTModel(model="gpt-3.5-turbo") # Free Quota exhausted for this model. So using ollama. 
# answer_relevancy_metric = AnswerRelevancyMetric() # It will select model automatically based on Open AI API key or Ollama. 

ollama_model = OllamaModel(model="llama3.2:latest")

answer_relevancy_metric = AnswerRelevancyMetric(model=ollama_model)

test_case = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

answer_relevancy_metric.measure(test_case)
print(answer_relevancy_metric.score)

In [ ]:
# Original Code below if you are running with OpenAI Moddel. 
# from deepeval.test_case import LLMTestCase
# from deepeval.metrics import AnswerRelevancyMetric

# answer_relevancy_metric = AnswerRelevancyMetric() # It will select model automatically based on Open AI API key or Ollama. 

# test_case = LLMTestCase(
#   input="Who is the current president of the United States of America?",
#   actual_output="Joe Biden",
#   retrieval_context=["Joe Biden serves as the current president of America."]
# )

# answer_relevancy_metric.measure(test_case)
# print(answer_relevancy_metric.score)

- input → the question asked to the LLM
- actual_output → the LLM’s answer
- retrieval_context → the information your system retrieved to help answer the question.
- It will select the LLM model automatically from the API key and other params. 

##### DeepEval’s Answer Relevancy score usually ranges from:
- 0 → 1
- Where:
    - 1.0 = Perfect relevance: The answer is completely supported by the retrieved context and matches the question.
    - 0.8 – 1.0 = Very good: Mostly relevant, minor wording differences.
    - 0.5 – 0.8 = Medium: Partly correct or partially related, but missing details.
    - 0.2 – 0.5 = Poor: Mostly irrelevant or only slightly connected.
    - 0.0 – 0.2 = Very bad: No relevance; the answer is unrelated or hallucinated.

### Test using Contextual Precision Metrics - Standalone
- A metric that evaluates how precisely the answer matches the context provided (important in RAG/agent workflows)
- success → Boolean flag (True if the answer is considered contextually precise enough).
- score_breakdown → Detailed explanation of how the score was calculated (e.g., token‑level precision, context alignment).


In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric
from deepeval.models import OllamaModel
ollama_model = OllamaModel(model="llama3.2:latest")

# contextual_precision_metrics = ContextualPrecisionMetric()
contextual_precision_metrics = ContextualPrecisionMetric(model = ollama_model)

test_case = LLMTestCase(
    input="Who is the current president of USA in 2024",
    # Should come from an LLM or from an Agent or RAG
    actual_output="Donald Trump",
    # RAG - Vector DB, AI Agent - Agent Tools, LLM - LLM invoke response
    retrieval_context=["Donald Trump serves as the current president of America."],
    expected_output="Donald Trump is the current president of America."
)

contextual_precision_metrics.measure(test_case=test_case)
print(contextual_precision_metrics.score)
print(contextual_precision_metrics.success)
print(contextual_precision_metrics.score_breakdown)



### Evaluate our Tests without Standalone using - Evaluate

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

ollama_model = OllamaModel(model="llama3.2:latest")
answer_relevancy_metric = AnswerRelevancyMetric(model=ollama_model)

test_case = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden", # This will be coming from the LLM, RAG or Agent. But currently we are hardcoding it for demo purpose.
  retrieval_context=["Joe Biden serves as the current president of America."]
)

evaluate(test_cases=[test_case], metrics=[answer_relevancy_metric])

# this evaluate method will send the result to Confident AI Portal
# It will automatically open the link like this https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/test-runs/cmiuhj50q0begpg1ffalfkmj1/test-cases?pageNumber=1&pageSize=50&status=all

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

# answer_relevancy_metric = AnswerRelevancyMetric()
ollama_model = OllamaModel(model="llama3.2:latest")
answer_relevancy_metric = AnswerRelevancyMetric(model=ollama_model)

test_case1 = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

test_case2 = LLMTestCase(
  input="Who built the Claude Models?",
  actual_output="OpenAITest", 
  expected_output= "Claude Anthrophic",
  retrieval_context=["Claude Anthrophic built the GPT models."]
)

evaluate(test_cases=[test_case1, test_case2], metrics=[answer_relevancy_metric])
# We can also compare the results of multiple models by passing different model instances to the in Confident AI Portal.

### Evaluate With Golden DataSet and EvaluationDataSet

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

# Create Golden instead of Test cases
golden = Golden(
    input="Who is the current president of the United States of America?",
    expected_output="Joe Biden",
    context=["Joe Biden serves as the current president of America."] # Only diff from Test case is that we use 'context' instead of 'retrieval_context' in Golden.
)

dataset = EvaluationDataset()
dataset.add_golden(golden)



In [ ]:
dataset

#### Creating Test Case from Golden

In [ ]:
from deepeval.models import OllamaModel
ollama_model = OllamaModel(model="llama3.2:latest")
# Above code is to setup the Ollama model for evaluation, otherwise it will use OpenAI model based on the API key. So in that case, we wont need above two lines. 

for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,
        expected_output=golden.expected_output,
        actual_output="Joe Biden",
        retrieval_context=golden.context
    )
    
    dataset.add_test_case(test_case)
    
evaluate(test_cases=dataset.test_cases, metrics=[AnswerRelevancyMetric(model = ollama_model)])

#### Creating Evaluation Dataset as Goldens in Confident AI 

###### Creating Data

In [2]:
test_data = [
    {
        "input": "Who is the current president of the United States of America?",
        "expected_output": "Joe Biden",
    },
    {
        "input": "Who introducted the GPT Model?",
        "expected_output": "Open AI"
    }
]

In [3]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

goldens = []

for data in test_data:
    golden = Golden(
        input= data['input'],
        expected_output=data['expected_output'],
    )
    goldens.append(golden)
    
new_dataset = EvaluationDataset(goldens=goldens)
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

#### Push the Dataset to Confident AI

In [4]:
new_dataset.push(alias="TestGoldenDataSet", finalized=False)

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=923371;https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/datasets/cmiv9frhl01p6n11ftkn0x8x1\https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/datasets/cmiv9frhl01p6n11ftkn0x8x1]8;;\

- the finalized parameter controls whether the dataset is considered ready for evaluation and locked from further edits.

In [5]:
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

#### Pull the Dataset from Confident AI


In [7]:
cloudDataSet = EvaluationDataset()
cloudDataSet.pull(alias="TestGoldenDataSet")
cloudDataSet

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=TestGoldenDataSet, _id=cmiv9frhl01p6n11ftkn0x8x1, _multi_turn=False)

- Confident AI checks if the dataset with that alias has any finalized golden entries. If it's empty (e.g., all entries were deleted or never added), it throws this error.
- finalised test data also can be edited directly from the Confident AI. 

#### Prepare our Testcase to evaluate our records

In [8]:
def mock_llms_app(input):
    if input == 1:
        return "Joe Biden"
    elif input == 2:
        return "Open AI"

In [9]:
from deepeval.test_case import LLMTestCase

counter = 1
for golden in cloudDataSet.goldens:
    test_case = LLMTestCase(
        input= golden.input,
        expected_output=golden.expected_output,
        actual_output=mock_llms_app(counter),
    )
    counter += 1
    cloudDataSet.add_test_case(test_case)

In [10]:
print(cloudDataSet.test_cases)

[LLMTestCase(input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None), LLMTestCase(input='Who introducted the GPT Model?', actual_output='Open AI', expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)]


In [13]:
from deepeval.models import OllamaModel
ollama_model = OllamaModel(model="llama3.2:latest")
evaluate(test_cases=cloudDataSet.test_cases, metrics=[AnswerRelevancyMetric(model = ollama_model)])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:latest (Ollama), strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Answer Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: llama3.2:latest (Ollama), reason: The score is 0.50 because the answer provided does not address the question about the current president of the United States of America, and instead only mentions a specific individual without providing relevant information about the current president., error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: Joe Biden
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 0.625, threshold: 0.5, strict: False, evaluation model: llama3.2:latest (Ollama), reason: The score is 0.62 because most of the output is about the laptop's features, which are not relevant to answering who introduced the GPT model. However, a small portion of the text mentions 'GPT', indicating some relevance, but overall, the majority of the out

⚠ WARNING: No hyperparameters logged.
» ]8;id=442510;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=488572;https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/test-runs/cmiva4su903g8pg1fl4oxown0/test-cases\https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/test-runs/cmiva4su903g8pg1fl4oxown0/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.5, reason='The score is 0.50 because the answer provided does not address the question about the current president of the United States of America, and instead only mentions a specific individual without providing relevant information about the current president.', strict_mode=False, evaluation_model='llama3.2:latest (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "no",\n        "reason": "The statement only provides information about a specific individual, without addressing the broader context of the current president."\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', e

### Using Local LLM for Evaluation

In [15]:
!deepeval set-ollama qwen2.5:latest
# It will add a line in the deepeval config file to use qwen2.5 model from Ollama as default model.

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `qwen2.5:latest` for 
all evals that require an LLM.


In [20]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric()

test_case1 = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

test_case2 = LLMTestCase(
  input="Who built the Claude Models?",
  actual_output="OpenAI",
  expected_output= "Claude Anthrophic",
  retrieval_context=["Claude Anthrophic built the GPT models."]
)

evaluate(test_cases=[test_case1, test_case2], metrics=[answer_relevancy_metric])

# This approach is not working, so we can use the same approach used above. 

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:8b (Ollama), strict=False, 
async_mode=True)...

ResponseError: model 'deepseek-r1:8b' not found (status code: 404)